# Análisis exploratorio de la presión de aparcamiento en Madrid (interior de la M-30)

Este cuaderno reúne el análisis exploratorio que sustenta el prototipo **SmartPark Madrid**.
A partir de los datos abiertos del Servicio de Estacionamiento Regulado (SER) del Ayuntamiento
de Madrid, estudio la demanda de aparcamiento (tickets), la oferta (plazas y parquímetros) y la
relación entre ambas para los siete distritos situados en el interior de la M-30.

Trabajo siempre sobre el ámbito M-30 porque es el alcance real del prototipo. Las cifras que
aparecen aquí son las mismas que utilizo en la memoria.

**Fuentes:**
- `data/tickets_m30.parquet` — tickets del SER ya filtrados al interior de la M-30 (demanda).
- `datasets/218228-...xlsx` — calles y plazas del SER (oferta).
- `datasets/300481-...xlsx` — parquímetros del SER.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

AZUL = "#3182bd"
GRIS = "#bdbdbd"

# Separador de miles (formato espanol) para el eje Y
_MILES = FuncFormatter(lambda x, _: f"{x:,.0f}".replace(",", "."))


def formato_miles(ax):
    ax.yaxis.set_major_formatter(_MILES)
    ax.spines[["top", "right"]].set_visible(False)


def encontrar_raiz():
    "Localiza la raiz del repositorio buscando data/tickets_m30.parquet."
    for cand in [Path.cwd(), *Path.cwd().parents]:
        if (cand / "data" / "tickets_m30.parquet").exists():
            return cand
    raise FileNotFoundError("No se encuentra data/tickets_m30.parquet")


RAIZ = encontrar_raiz()
print("Raiz del proyecto:", RAIZ)

## 1. Carga de datos

Cargo la demanda (tickets) y la oferta (plazas y parquímetros), filtrando estas últimas a los
distritos presentes en los tickets de la M-30.

In [ ]:
tickets = pd.read_parquet(RAIZ / "data" / "tickets_m30.parquet")
distritos_m30 = tickets["distrito"].unique()
total_tickets = len(tickets)

plazas = pd.read_excel(RAIZ / "datasets" / "218228-0-ser-calles SER CALLES Y PLAZAS-xlsx.xlsx")
plazas = plazas[plazas["distrito"].isin(distritos_m30)]

parquimetros = pd.read_excel(RAIZ / "datasets" / "300481-3-ser-parquimetros-xlsx.xlsx")
parquimetros = parquimetros[parquimetros["distrito"].isin(distritos_m30)]

print(f"Tickets procesados (M-30): {total_tickets:,}")
print(f"Distritos ({len(distritos_m30)}): {sorted(distritos_m30)}")
tickets.head()

## 2. Distribución horaria de la demanda

Agrupo los tickets por hora del día para identificar las franjas de mayor presión.

In [ ]:
por_hora = tickets.groupby("hora").size()
pct_hora = por_hora / total_tickets * 100

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(por_hora.index, por_hora.values, color=AZUL)
ax.set_title("Distribución horaria de los tickets (interior de la M-30)")
ax.set_xlabel("Hora del día")
ax.set_ylabel("Número de tickets")
ax.set_xticks(range(int(por_hora.index.min()), int(por_hora.index.max()) + 1))
formato_miles(ax)
plt.show()

print(f"Pico: {pct_hora.idxmax()}h ({pct_hora.max():.1f}%)")
print(f"Franja 9-14h: {pct_hora[(pct_hora.index >= 9) & (pct_hora.index <= 14)].sum():.1f}%")
print(f"Franja 17-20h: {pct_hora[(pct_hora.index >= 17) & (pct_hora.index <= 20)].sum():.1f}%")
print(f"Valle 15h: {pct_hora.get(15, 0):.1f}%")

La demanda se concentra por la mañana, con el máximo a primera hora y un descenso al mediodía.
No aparece el clásico doble pico mañana-tarde: la franja de tarde pesa menos y la actividad
prácticamente desaparece a partir de las 21h.

## 3. Demanda por distrito

Cuento los tickets por distrito para ver dónde se concentra la presión.

In [ ]:
dem = tickets.groupby("distrito").size().sort_values(ascending=False)
pct_dem = (dem / total_tickets * 100).round(1)

fig, ax = plt.subplots(figsize=(9, 5))
barras = ax.bar([d.capitalize() for d in dem.index], dem.values, color=AZUL)
ax.set_title("Demanda por distrito (interior de la M-30)")
ax.set_ylabel("Número de tickets")
ax.set_ylim(0, dem.max() * 1.12)
for ba, d in zip(barras, dem.index):
    ax.text(ba.get_x() + ba.get_width() / 2, ba.get_height(),
            f"{pct_dem[d]:.1f}%", ha="center", va="bottom",
            fontsize=10, fontweight="bold")
plt.setp(ax.get_xticklabels(), rotation=25, ha="right")
formato_miles(ax)
plt.show()

for d in dem.index:
    print(f"{d.capitalize():12s} {dem[d]:>9,}  {pct_dem[d]:5.1f}%")
print(f"Top-3: {pct_dem.head(3).sum():.1f}%")
print(f"Razón máximo/mínimo: {dem.iloc[0] / dem.iloc[-1]:.1f}x")

## 4. Niveles de presión por distrito (terciles)

Clasifico los distritos en tres niveles de presión (alta, media y baja) repartiendo la demanda
en terciles.

In [ ]:
niveles = pd.qcut(dem, 3, labels=["Baja", "Media", "Alta"])
color_map = {"Alta": "#d62728", "Media": "#fd8d3c", "Baja": "#2ca25f"}

fig, ax = plt.subplots(figsize=(9, 5))
barras = ax.bar([d.capitalize() for d in dem.index], dem.values,
                color=[color_map[niveles[d]] for d in dem.index])
ax.set_title("Presión por distrito segmentada en terciles (interior de la M-30)")
ax.set_ylabel("Número de tickets")
ax.set_ylim(0, dem.max() * 1.12)
for ba, d in zip(barras, dem.index):
    ax.text(ba.get_x() + ba.get_width() / 2, ba.get_height(),
            f"{pct_dem[d]:.1f}%", ha="center", va="bottom",
            fontsize=10, fontweight="bold")
plt.setp(ax.get_xticklabels(), rotation=25, ha="right")
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in color_map.values()]
ax.legend(handles, color_map.keys(), title="Presión")
formato_miles(ax)
plt.show()

for nivel in ["Alta", "Media", "Baja"]:
    ds = [d.capitalize() for d in dem.index if niveles[d] == nivel]
    print(f"{nivel}: {len(ds)} -> {ds}")

## 5. Tipo de zona: oferta frente a demanda

Comparo el reparto de plazas (oferta) y de tickets (demanda) entre zona Verde, Azul y otras.

In [ ]:
plazas["cn"] = plazas["color"].str.split().str[1:].str.join(" ").str.upper()
pl = plazas.groupby("cn")["numero_plazas"].sum()
pl_v, pl_a = int(pl.get("VERDE", 0)), int(pl.get("AZUL", 0))
pl_o = int(pl.sum() - pl_v - pl_a)

tk = tickets.groupby("tipo_zona").size()
tk_v, tk_a = int(tk.get("VERDE", 0)), int(tk.get("AZUL", 0))
tk_o = int(tk.sum() - tk_v - tk_a)

cats = ["Verde", "Azul", "Otros"]
pv = [pl_v, pl_a, pl_o]
tv = [tk_v, tk_a, tk_o]
pp = [v / sum(pv) * 100 for v in pv]
tp = [v / sum(tv) * 100 for v in tv]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
cols = ["#2ca25f", AZUL, GRIS]
b = ax1.bar(cats, pv, color=cols, width=0.6)
ax1.set_title("Oferta: número de plazas por tipo")
ax1.set_ylabel("Número de plazas")
ax1.set_ylim(0, max(pv) * 1.15)
for ba, va in zip(b, pv):
    ax1.text(ba.get_x() + ba.get_width() / 2, va, f"{va:,.0f}".replace(",", "."),
             ha="center", va="bottom", fontsize=10, fontweight="bold")
formato_miles(ax1)

x = np.arange(len(cats))
w = 0.38
b1 = ax2.bar(x - w / 2, pp, w, label="% de plazas (oferta)", color="#a1d99b")
b2 = ax2.bar(x + w / 2, tp, w, label="% de tickets (demanda)", color="#08519c")
ax2.set_title("Oferta vs demanda (en %)")
ax2.set_ylabel("Porcentaje sobre el total")
ax2.set_xticks(x)
ax2.set_xticklabels(cats)
ax2.set_ylim(0, 100)
ax2.legend()
for bars in (b1, b2):
    for ba in bars:
        ax2.text(ba.get_x() + ba.get_width() / 2, ba.get_height(),
                 f"{ba.get_height():.1f}%", ha="center", va="bottom", fontsize=9)
ax2.spines[["top", "right"]].set_visible(False)
plt.show()

print(f"Plazas: Verde {pl_v:,} ({pp[0]:.1f}%) | Azul {pl_a:,} ({pp[1]:.1f}%) | Otros {pl_o:,} ({pp[2]:.1f}%)")
print(f"Tickets: Verde {tp[0]:.1f}% | Azul {tp[1]:.1f}% | Otros {tp[2]:.1f}%")
print(f"Total plazas M-30: {int(pl.sum()):,}")

La zona Verde domina tanto la oferta como la demanda, lo que refleja el peso de los residentes
sobre la disponibilidad de plaza para quienes están de paso.

## 6. Oferta de plazas por distrito

In [ ]:
pl_dist = plazas.groupby("distrito")["numero_plazas"].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar([d.capitalize() for d in pl_dist.index], pl_dist.values, color=AZUL)
ax.set_title("Plazas SER por distrito (interior de la M-30)")
ax.set_ylabel("Número de plazas")
plt.setp(ax.get_xticklabels(), rotation=25, ha="right")
formato_miles(ax)
plt.show()

print(f"Más plazas: {pl_dist.index[0].capitalize()} {pl_dist.iloc[0]:,}")
print(f"Menos plazas: {pl_dist.index[-1].capitalize()} {pl_dist.iloc[-1]:,}")

## 7. Parquímetros por distrito

Reviso el reparto de parquímetros y su relación con el número de plazas.

In [ ]:
pq_dist = parquimetros.groupby("distrito").size().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar([d.capitalize() for d in pq_dist.index], pq_dist.values, color=AZUL)
ax.set_title("Parquímetros por distrito (interior de la M-30)")
ax.set_ylabel("Número de parquímetros")
plt.setp(ax.get_xticklabels(), rotation=25, ha="right")
formato_miles(ax)
plt.show()

comb = pd.DataFrame({"plazas": pl_dist, "parq": pq_dist}).dropna()
r = comb["plazas"].corr(comb["parq"])
print(f"Correlación plazas-parquímetros: r = {r:.2f}")
print(f"Total parquímetros: {len(parquimetros):,}")

## 8. Presión relativa: tickets por plaza

Divido la demanda entre la oferta de cada distrito para obtener una medida de presión que no
depende del tamaño del distrito.

In [ ]:
ratio = (dem / pl_dist).dropna().sort_values(ascending=False)
mediana = ratio.median()
cols = []
for d in ratio.index:
    if ratio[d] == ratio.max():
        cols.append("#d62728")
    elif ratio[d] == ratio.min():
        cols.append("#2ca25f")
    else:
        cols.append(AZUL)

fig, ax = plt.subplots(figsize=(9, 5))
barras = ax.bar([d.capitalize() for d in ratio.index], ratio.values, color=cols)
ax.axhline(mediana, color="#636363", ls="--", lw=1.5)
ax.text(len(ratio) - 0.5, mediana + 1.5, f"Mediana: {mediana:.1f}", ha="right",
        color="#636363", fontstyle="italic")
for ba, va in zip(barras, ratio.values):
    ax.text(ba.get_x() + ba.get_width() / 2, va, f"{va:.1f}",
            ha="center", va="bottom", fontsize=10, fontweight="bold")
ax.set_title("Presión relativa por distrito: tickets por plaza (interior de la M-30)")
ax.set_ylabel("Tickets por plaza (trimestre)")
plt.setp(ax.get_xticklabels(), rotation=25, ha="right")
ax.set_ylim(0, ratio.max() * 1.15)
ax.spines[["top", "right"]].set_visible(False)
plt.show()

for d in ratio.index:
    print(f"{d.capitalize():12s} {ratio[d]:6.1f}")
print(f"Mediana: {mediana:.1f} | Máx: {ratio.idxmax().capitalize()} {ratio.max():.1f} | "
      f"Mín: {ratio.idxmin().capitalize()} {ratio.min():.1f}")

## 9. Comparación normalizada entre oferta y demanda

Escalo oferta y demanda a un rango 0-1 para compararlas directamente por distrito.

In [ ]:
dfn = pd.DataFrame({"demanda": dem, "plazas": pl_dist}).dropna()
dfn["dn"] = dfn["demanda"] / dfn["demanda"].max()
dfn["pn"] = dfn["plazas"] / dfn["plazas"].max()
dfn = dfn.sort_values("demanda", ascending=False)

x = np.arange(len(dfn))
w = 0.38
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - w / 2, dfn["dn"], w, label="Demanda (normalizada)", color="#08519c")
ax.bar(x + w / 2, dfn["pn"], w, label="Oferta de plazas (normalizada)", color="#a1d99b")
ax.set_xticks(x)
ax.set_xticklabels([d.capitalize() for d in dfn.index], rotation=25, ha="right")
ax.set_title("Comparación normalizada entre oferta y demanda (interior de la M-30)")
ax.set_ylabel("Nivel relativo (0-1)")
ax.legend()
ax.spines[["top", "right"]].set_visible(False)
plt.show()

## 10. Distribución por distintivo ambiental

In [ ]:
dist = (tickets.groupby("distintivo").size() / total_tickets * 100).round(1).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.bar(dist.index.astype(str), dist.values, color=AZUL)
ax.set_title("Distribución de tickets por distintivo ambiental (interior de la M-30)")
ax.set_ylabel("Porcentaje sobre el total")
for i, v in enumerate(dist.values):
    ax.text(i, v, f"{v:.1f}%", ha="center", va="bottom", fontsize=10, fontweight="bold")
ax.spines[["top", "right"]].set_visible(False)
plt.show()

print(dist.to_string())

## 11. Conclusiones del análisis

- La demanda se concentra por la mañana, sin doble pico claro, y cae a partir de las 21h.
- Geográficamente la presión está muy desequilibrada: los distritos del norte concentran la
  mayor parte de los tickets, frente a Centro, que es el de menor demanda dentro de la M-30.
- La zona Verde domina oferta y demanda, lo que limita la rotación disponible.
- La presión relativa (tickets por plaza) reordena el mapa: un distrito con muchas plazas no es
  necesariamente el más tensionado.

Estos patrones justifican un sistema que estime la probabilidad de encontrar plaza combinando
distrito, hora, tipo de zona y distintivo, que es exactamente lo que implementa el prototipo
SmartPark Madrid.